In [3]:
import sqlite3
from pathlib import Path
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from sklearn.model_selection import train_test_split
from dataset import SoccerDataset

# --- 1. Load Data ---
db_path = Path("../../data/database.sqlite")
conn = sqlite3.connect(db_path.as_posix())
df = pd.read_sql_query("SELECT * FROM Player_Formation_Preprocessed", conn)
conn.close()

player_stats_cols = [
    "preferred_foot",
    "attacking_work_rate",
    "defensive_work_rate",
    "crossing",
    "finishing",
    "heading_accuracy",
    "short_passing",
    "volleys",
    "dribbling",
    "curve",
    "free_kick_accuracy",
    "long_passing",
    "ball_control",
    "acceleration",
    "sprint_speed",
    "agility",
    "reactions",
    "balance",
    "shot_power",
    "jumping",
    "stamina",
    "strength",
    "long_shots",
    "aggression",
    "interceptions",
    "positioning",
    "vision",
    "penalties",
    "marking",
    "standing_tackle",
    "sliding_tackle",
]

# Extract x, y and player stats
positions = torch.tensor(df[["x", "y"]].to_numpy(), dtype=torch.float32)
players = torch.tensor(df[player_stats_cols].to_numpy(), dtype=torch.float32)

# --- 2. Model Architecture ---


class PositionTower(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(in_features=2, out_features=64),
            nn.BatchNorm1d(64),
            nn.LeakyReLU(),
            nn.Linear(in_features=64, out_features=128),
            nn.BatchNorm1d(128),
            nn.LeakyReLU(),
            nn.Linear(in_features=128, out_features=64),
        )

    def forward(self, x):
        return self.net(x)


class PlayerTower(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(in_features=31, out_features=128),
            nn.BatchNorm1d(128),
            nn.LeakyReLU(),
            nn.Linear(in_features=128, out_features=128),
            nn.BatchNorm1d(128),
            nn.LeakyReLU(),
            nn.Linear(in_features=128, out_features=64),
        )

    def forward(self, x):
        return self.net(x)


class SimpleTwoTower(nn.Module):
    def __init__(self):
        super().__init__()
        self.position_tower = PositionTower()
        self.player_tower = PlayerTower()

    def forward(self, position, player):
        pos_emb = self.position_tower(position)
        plr_emb = self.player_tower(player)

        p_norm = pos_emb / (torch.linalg.norm(pos_emb, dim=1, keepdim=True) + 1e-8)
        u_norm = plr_emb / (torch.linalg.norm(plr_emb, dim=1, keepdim=True) + 1e-8)

        return p_norm, u_norm


# --- 3. Dataset & Loaders ---
dummy_formations = torch.zeros((len(positions), 9, 2))

indices = np.arange(len(players))
train_idx, temp_idx = train_test_split(indices, test_size=0.2, random_state=42)
val_idx, test_idx = train_test_split(temp_idx, test_size=0.5, random_state=42)

train_dataset = SoccerDataset(
    dummy_formations[train_idx], positions[train_idx], players[train_idx]
)
val_dataset = SoccerDataset(
    dummy_formations[val_idx], positions[val_idx], players[val_idx]
)
test_dataset = SoccerDataset(
    dummy_formations[test_idx], positions[test_idx], players[test_idx]
)

train_loader = DataLoader(
    train_dataset, batch_size=512, shuffle=True, num_workers=4, pin_memory=True
)
val_loader = DataLoader(
    val_dataset, batch_size=1024, shuffle=False, num_workers=4, pin_memory=True
)

# --- 4. Training & Validation Functions ---


def train_one_epoch(loader, model, optimizer, criterion, device, temp=0.07):
    model.train()
    total_loss = 0
    for pos, _, player in loader:
        pos, player = pos.to(device), player.to(device)

        p_emb, u_emb = model(pos, player)
        logits = torch.matmul(p_emb, u_emb.T) / temp
        labels = torch.arange(pos.size(0)).to(device)
        loss = criterion(logits, labels)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    return total_loss / len(loader)


def validate(loader, model, criterion, device, temp=0.07):
    model.eval()
    total_loss = 0
    with torch.no_grad():
        for pos, _, player in loader:
            pos, player = pos.to(device), player.to(device)
            p_emb, u_emb = model(pos, player)
            logits = torch.matmul(p_emb, u_emb.T) / temp
            labels = torch.arange(pos.size(0)).to(device)
            loss = criterion(logits, labels)
            total_loss += loss.item()
    return total_loss / len(loader)


# --- 5. Main Execution ---

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = SimpleTwoTower().to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
criterion = nn.CrossEntropyLoss()

num_epochs = 200
best_val_loss = float("inf")

print(f"Starting training on {device}...")

for epoch in range(num_epochs):
    train_loss = train_one_epoch(train_loader, model, optimizer, criterion, device)
    val_loss = validate(val_loader, model, criterion, device)

    print(
        f"Epoch {epoch + 1}/{num_epochs} | Train Loss: {train_loss:.4f} | Val Loss: {val_loss:.4f}"
    )

    # Only save the best model based on validation performance
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        torch.save(model.state_dict(), "simple_soccer_model.pth")
        print(f"--> Best model saved with Val Loss: {val_loss:.4f}")

Starting training on cuda...
Epoch 1/200 | Train Loss: 5.2089 | Val Loss: 5.8299
--> Best model saved with Val Loss: 5.8299
Epoch 2/200 | Train Loss: 5.1154 | Val Loss: 5.7896
--> Best model saved with Val Loss: 5.7896
Epoch 3/200 | Train Loss: 5.0682 | Val Loss: 5.7437
--> Best model saved with Val Loss: 5.7437
Epoch 4/200 | Train Loss: 5.0331 | Val Loss: 5.7202
--> Best model saved with Val Loss: 5.7202
Epoch 5/200 | Train Loss: 5.0037 | Val Loss: 5.6959
--> Best model saved with Val Loss: 5.6959
Epoch 6/200 | Train Loss: 4.9787 | Val Loss: 5.6786
--> Best model saved with Val Loss: 5.6786
Epoch 7/200 | Train Loss: 4.9591 | Val Loss: 5.6593
--> Best model saved with Val Loss: 5.6593
Epoch 8/200 | Train Loss: 4.9417 | Val Loss: 5.6471
--> Best model saved with Val Loss: 5.6471
Epoch 9/200 | Train Loss: 4.9292 | Val Loss: 5.6390
--> Best model saved with Val Loss: 5.6390
Epoch 10/200 | Train Loss: 4.9175 | Val Loss: 5.6295
--> Best model saved with Val Loss: 5.6295
Epoch 11/200 | Train

KeyboardInterrupt: 